# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import os
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
logging.info(f'Path [{os.environ["PATH"]}]')

# Test pytorch GPU config
import torch
cuda_test = torch.cuda.is_available()
logging.info(f"✅ Torch CUDA available: {cuda_test}")
device_name = torch.cuda.get_device_name(0)
torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logging.info(f"🖥️ Device Name: {device_name} | Device reference: {torch_device.type}")

### machine learning (scikit-learn)
import math
import numpy as np
import pandas as pd
from typing import cast
from sklearn.pipeline import Pipeline
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import warnings
warnings.filterwarnings(
    "ignore",
    message="mtime may not be reliable on this filesystem, falling back to numerical ordering"
)
from transformers import (
    EarlyStoppingCallback, Trainer, TrainingArguments, set_seed  # type: ignore
)
from tsfm_public import (
    TinyTimeMixerForPrediction,
    TimeSeriesForecastingPipeline,
    TrackingCallback,
    count_parameters,
)
from tsfm_public.toolkit.time_series_preprocessor import get_datasets, prepare_data_splits
from tsfm_public.toolkit.visualization import plot_predictions
from tsfm_public.toolkit.service_util import save_deployment_package

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Project Specific imports

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.preprocessing_project_specific as pps
import smartcheck.deep_learning_project_specific as dlps

# 2. Loading and Preprocessing

## 2.1 Loading data

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Data Refactoring pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    # "date_et_heure_de_comptage",
    "date_et_heure_de_comptage_local",
    # "date_et_heure_de_comptage_utc",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("convert_datetime", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage",
                                                                   for_sarimax=True)),
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

# 3. Modeling Training and Prediction

## 3.1 Contextual variables

#### For the model

In [ ]:
# Model inner parameters 
context_length = 512 # max possible for this model
fcm_context_length = 48 # 1 lag = 1 hour
prediction_length = 96 # max possible for this model
OUT_DIR = "ttm_results.model" # model runtime and training data archiving/export.
# Parameters for the training
learning_rate: float = 0.0002
num_epochs: int = 200
patience: int = 10
batch_size: int = 96
steps_per_epoch=math.ceil(10000 / batch_size)
# Parameters for the preprocessing
fewshot_fraction = 1 # fasten training by returning a percent original train dataset
set_seed(42)
split_config = {"train": 0.6, "test": 0.2}
timestamp_column = "date_et_heure_de_comptage_local"
target_columns = ["comptage_horaire"]
column_specifiers = {
    "timestamp_column": timestamp_column,
    "id_columns": [
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ],
    "target_columns": target_columns,
    "categorical_columns": [
        "vacances_scolaires",
        "weather_code_wmo_code_category",
    ],
    "control_columns": [
        "jour_ferie",
        "vacances_scolaires",
    ],
    "observable_columns": [
        "temperature_2m_c",
        "rain_mm",
        "snowfall_cm",
        "weather_code_wmo_code_category",
    ],
}

#### For the experiments

In [ ]:
dict_compteurs = {
    "experiment_1": {
        "name": "Subset_Counters_Study_v1",
        "counters_dict": {
            ('Totem 73 boulevard de Sébastopol', 'S-N'): "Sébastopol_S-N",
            ('Totem 73 boulevard de Sébastopol', 'N-S'): "Sébastopol_N-S",
            ('102 boulevard de Magenta', 'SE-NO'): "Magenta_SE-NO",
            ('Pont de Bercy', 'NE-SO'): "Bercy_NE-SO",
            ('Pont de Bercy', 'NE-SO'): "Bercy_NE-SO",
            ('135 avenue Daumesnil', 'SE-NO'): "Daumesnil_SE-NO",
            ("180 avenue d'Italie", 'N-S'): "Italie_N-S",
            ('27 quai de la Tournelle', 'NO-SE'): "Tournelle_NO-SE",
            ('27 quai de la Tournelle', 'SE-NO'): "Tournelle_SE-NO",
        },
        "sub_range": (0,),
        "out_dir": OUT_DIR,            
        "context_length": context_length,
        "fcm_context_length": fcm_context_length,
        "prediction_length": prediction_length,
        # "best_checkpoint": "checkpoint-7",
    },
}

## 3.2 Data Viz of Time Series per counter

In [ ]:
# Not Applicable

## 3.3 Transfert learning with preprocessing and training

#### Définition et ajustement du modèle granite pour finetuning avec gel des couches pré-entrainées
> Environ 500k paramètres gelés mais il reste ceux ajoutés via notre contexte de variables exogènes, le nombre de paramètres additionnels dépend des variables exogènes et de la fenetre de contexte (FCM)
> - 24 lags (1 jour) -> environ 900k paramètres
> - 48 lags (2 jours) -> environ 2,7M paramètres
> - 168 lags (7 jours) -> environ 29M de paramètres

In [ ]:
def fine_tune_model(output_dir, logging_dir):
    # Define preprocessor
    finetune_timeseries_preprocessor = dlps.SafeTimeSeriesPreprocessorOrdinal(
        **column_specifiers,
        context_length=context_length,
        prediction_length=prediction_length,
        scaling=True,
        freq="h",
        encode_categorical=True,
        scale_categorical_columns=True,
        scaler_type="standard",  # type: ignore
    )

    # Define model from Hugging Face
    finetune_forecast_model = TinyTimeMixerForPrediction.from_pretrained(
        "ibm-granite/granite-timeseries-ttm-r2",  # Name of the model on HuggingFace.
        num_input_channels=finetune_timeseries_preprocessor.num_input_channels,
        prediction_channel_indices=finetune_timeseries_preprocessor.prediction_channel_indices,
        exogenous_channel_indices=finetune_timeseries_preprocessor.exogenous_channel_indices,
        fcm_use_mixer=True,
        fcm_context_length=fcm_context_length,  
        enable_forecast_channel_mixing=True,
        decoder_mode="mix_channel",
    )
    logging.info(f"Bascule du modèle sur {torch_device}")
    finetune_forecast_model.to(torch_device)  # type: ignore

    # Freeze the backbone of the model
    logging.info(f"Number of params before freezing backbone {count_parameters(finetune_forecast_model)}")
    for param in finetune_forecast_model.backbone.parameters():
        param.requires_grad = False
    logging.info(f"Number of params after freezing the backbone {count_parameters(finetune_forecast_model)}")

    # Set the training arguments
    logging.info(f"Learning Rate = {learning_rate} | {num_epochs} epoch(s) |"
                 f" utilisation du {'GPU' if (torch_device.type == 'cuda') else 'CPU'}")
    finetune_forecast_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        learning_rate=learning_rate,
        num_train_epochs=num_epochs,
        do_eval=True,
        eval_strategy="epoch",
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        dataloader_pin_memory=True,
        report_to=None,
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        logging_dir=logging_dir,  # Make sure to specify a logging directory
        load_best_model_at_end=True,  # Load the best model when training ends
        metric_for_best_model="eval_loss",  # Metric to monitor for early stopping
        greater_is_better=False,  # For loss
        no_cuda=torch_device.type != "cuda",
    )

    # Create the early stopping callback
    early_stopping_callback = EarlyStoppingCallback(
        early_stopping_patience=patience,  # Number of epochs with no improvement after which to stop
        early_stopping_threshold=0.00001,  # Minimum improvement required to consider as improvement
    )
    tracking_callback = TrackingCallback()

    # Define an optimizer and scheduler
    finetune_optimizer = AdamW(finetune_forecast_model.parameters(), lr=learning_rate)
    finetune_scheduler = OneCycleLR(
        finetune_optimizer,
        learning_rate,
        epochs=num_epochs,
        steps_per_epoch=steps_per_epoch,
    )
    return (
        finetune_timeseries_preprocessor,
        finetune_forecast_model,
        finetune_forecast_args,
        finetune_optimizer, finetune_scheduler,
        early_stopping_callback,
        tracking_callback
    )

#### Boucle d'entrainement du modèle

In [ ]:
model_results = {}
# Make a forecast on the target column given the input data.
for experiment, exp_params in dict_compteurs.items():
    # Filtrage de l'experience et de son dataset associé
    compteur_keys = list(exp_params["counters_dict"].keys())
    grouped_df = df.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    df_final = df[
        df[["nom_du_site_de_comptage", "orientation_compteur"]]
        .apply(tuple, axis=1)  # type: ignore
        .isin(compteur_keys)
    ].copy()
    sub_range = exp_params["sub_range"]
    range_start = sub_range[0]
    range_end = sub_range[1] if len(sub_range) > 1 else None
    df_final_sub = cast(pd.DataFrame, df_final[range_start:range_end])

    # Definition et fine tuning du modèle
    name = exp_params["name"]
    deploy_dir = os.path.join(OUT_DIR, f"deploy_{name}")
    preproc_dir = os.path.join(OUT_DIR, f"preproc_{name}")
    output_dir = os.path.join(OUT_DIR, f"output_{name}")
    logging_dir = os.path.join(OUT_DIR, f"log_{name}")
    (
        tsp,
        model,
        args,
        optimizer,
        scheduler,
        early_stop_cb,
        tracking_cb
    ) = fine_tune_model(output_dir, logging_dir)

    # Split train, valid, test for dataframes and datasets for training
    train_df, valid_df, test_df = prepare_data_splits(  # type: ignore
        df_final_sub,
        context_length=context_length,
        split_config=split_config  # type: ignore
    )
    logging.info(f"Dataframe lengths: train = {len(train_df)}, val = {len(valid_df)}, test = {len(test_df)}")
    train_dataset, valid_dataset, test_dataset = get_datasets(  # type: ignore
        tsp,
        df_final_sub,
        split_config,  # type: ignore
        stride=prediction_length,
        fewshot_fraction=fewshot_fraction,
        fewshot_location="first",
        use_frequency_token=model.config.resolution_prefix_tuning,
    )
    logging.info(f"Dataset batch lengths: train = {len(train_dataset)}, val = {len(valid_dataset)}, test = {len(test_dataset)}")

    # Définition du modèle et de son trainer
    finetune_forecast_trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        callbacks=[early_stop_cb, tracking_cb],
        optimizers=(optimizer, scheduler)  # type: ignore
    )

    # Priorité au reentrainement à partir du best checkpoint
    best_checkpoint = dlps.train_or_resume(finetune_forecast_trainer, exp_params)

    # sauvegarde des résultats en mémoire
    model_results[experiment] = {
        "exp_params": exp_params,
        "train_df": train_df,
        "valid_df": valid_df,
        "test_df": test_df,
        "best_checkpoint": best_checkpoint,
        "model": model,
        "tsp": tsp,
    }

    # sauvegarde sur disque de l'experience via joblib / yaml (preprocessor inclus)
    dlps.save_granite_model(experiment, model_results[experiment])
    dlps.save_preprocessor_state(tsp, preproc_dir)
    save_deployment_package(deploy_dir, model, ts_processor=tsp)

## 3.3 Predictions

#### From memory context

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_test = model_result["test_df"]
    model = model_result["model"]
    tsp = model_result["tsp"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    name = exp_params["name"]
    out_dir = exp_params["out_dir"]
    checkpoint_dir = os.path.join(out_dir, f"output_{name}", model_result['best_checkpoint'])
    preproc_dir = os.path.join(out_dir, f"preproc_{name}")
    
    # Create the evaluation pipeline
    pipeline = TimeSeriesForecastingPipeline(
        model=model,
        device=torch_device,
        feature_extractor=tsp,
        batch_size=batch_size,
    )

    # Collect and store the predictions
    predictions_df_test = pipeline(df_test)  # type: ignore
    model_results[experiment]["predictions_df_test"] = predictions_df_test

In [ ]:
for experiment, model_result in model_results.items():
    # Extract the artifacts from the saved results
    exp_params = model_result["exp_params"]
    df_test = model_result["test_df"]
    predictions_df_test = model_result["predictions_df_test"]
    logging.info(f"\n\nParamètres {exp_params} :\n")
    # Print/Plot the predictions
    grouped_df_test = df_test.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_df_test : [{len(grouped_df_test.groups.keys())}]")
    grouped_pred_df_test = predictions_df_test.groupby([
        "nom_du_site_de_comptage",
        "orientation_compteur",
    ])
    logging.info(f"Nombre de grouped_pred_df_test : [{len(grouped_pred_df_test.groups.keys())}]")
    for compteur_key, df_test in grouped_df_test:
        print(f"Prediction plot on each of the 3 previous week and in the future for {compteur_key}")
        df_test = df_test.sort_values(by=timestamp_column).reset_index()
        pred_df_test = grouped_pred_df_test.get_group(compteur_key).sort_values(by=timestamp_column).reset_index()
        plot_predictions(
            input_df=df_test,
            predictions_df=pred_df_test,  # type: ignore
            freq="h",
            timestamp_column=timestamp_column,
            channel=target_columns[0],
            # we check the prediction on each of the 3 previous week and in the future
            indices=[ -24*7*3, -24*7*2, -24*7*1, -1],
            num_plots=4,
        )
        plt.show()